In [ ]:

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import sqlite3

engine = sqlite3.connect('data/db/bluestock_mf.db')
df_nav = pd.read_sql('SELECT * FROM fact_nav', engine)
df_fund = pd.read_sql('SELECT * FROM dim_fund', engine)
df_aum = pd.read_sql('SELECT * FROM fact_aum', engine)
df_sip = pd.read_csv('data/raw/monthly_sip.csv')

# 1. NAV trend line charts (2022-2026)
df_nav_fund = df_nav.merge(df_fund, on='amfi_code')
fig1 = px.line(df_nav_fund, x='date_id', y='nav', color='scheme_name', title='NAV Trend (2022-2025)')
# Uncomment to show: fig1.show()

# 2. AUM growth bar charts
fig2 = px.bar(df_aum, x='year', y='aum_lakh_cr', color='amc', title='AUM Growth by AMC', barmode='group')
# Uncomment to show: fig2.show()

# 3. SIP inflow time-series
fig3 = px.line(df_sip, x='month', y='sip_amount_cr', title='SIP Inflows (2022-2025)', markers=True)
# Uncomment to show: fig3.show()

# 4. Correlation heatmap for returns
df_nav_pivot = df_nav_fund.pivot(index='date_id', columns='scheme_name', values='nav').pct_change().dropna()
# Take top 10 funds to make heatmap readable
top_10_funds = df_nav_fund['scheme_name'].unique()[:10]
corr_matrix = df_nav_pivot[top_10_funds].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Return Correlation Heatmap (Top 10 Funds)')
plt.tight_layout()
plt.savefig('notebooks/correlation_heatmap.png')
print("EDA plots generated and scripts ready. Note: Plotly output suppressed for non-interactive run.")
